# SavorLink: Cold-Start Food Pairing — GPU Benchmark on Colab

**Course:** Advanced Foundations of Machine Learning, PES University
**Hardware:** any Colab GPU runtime (tested target: free-tier Tesla T4). Runs on CPU too, just slower.

### What this notebook does
1. Unzips `colab_training_bundle.zip` (config, FlavorGraph data, precomputed features for all 4 modalities, and the `src/` code).
2. Checks the feature files for the four modalities: molecular, semantic, hybrid, taxonomic.
3. Runs `python -m src.run_benchmark`: trains Strategy III (GraphSAGE with pseudo-edges, BPR loss) and evaluates it against Random, Popularity, kNN and a zero-degree GNN, for 4 modalities x 3 seeds.
4. Prints the resulting table from `results/RESULTS.md`.
5. Zips `results/` (table and trained models) for download.

Create the bundle locally first: `python -m src.export_colab_bundle`.


--- 
## 1. Hardware Verification & GPU Environment Setup

In [ ]:
import torch
import os
import sys

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU:      {gpu_name} ({vram_gb:.2f} GB VRAM)")
    device = torch.device("cuda")
else:
    print("WARNING: CUDA not detected! In Colab, go to: Runtime -> Change runtime type -> T4 GPU")
    device = torch.device("cpu")
print("=" * 60)

# Install lightweight requirements
%pip install pyyaml scikit-learn scipy tqdm --quiet


--- 
## 2. Unzip the Training Bundle
Create the bundle locally with `python -m src.export_colab_bundle` (writes `data/colab_training_bundle.zip`, ~4 MB), then upload it to the Colab file panel.

In [ ]:
import os, sys, zipfile

bundle_path = "colab_training_bundle.zip"
if not os.path.exists(bundle_path) and os.path.exists("/content/colab_training_bundle.zip"):
    bundle_path = "/content/colab_training_bundle.zip"

if os.path.exists(bundle_path):
    print(f"Found bundle: {bundle_path} ({os.path.getsize(bundle_path) / 1e6:.2f} MB). Unzipping...")
    with zipfile.ZipFile(bundle_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Unzipped bundle successfully!")
else:
    print("Upload colab_training_bundle.zip in the Colab file browser on the left, then rerun this cell.")

# Ensure project root is in sys.path
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))


--- 
## 3. Verify Precomputed Features Across the 4 Modalities

In [ ]:
import numpy as np
import pickle
from pathlib import Path

modalities = {
    "molecular": {"feat": "data/cache/node_features.npy", "sim": "data/cache/topk_similarities.pkl"},
    "semantic":  {"feat": "data/cache/semantic_features.npy", "sim": "data/cache/semantic_similarities.pkl"},
    "hybrid":    {"feat": "data/cache/hybrid_features.npy", "sim": "data/cache/hybrid_similarities.pkl"},
    "taxonomic": {"feat": "data/cache/taxonomic_features.npy", "sim": "data/cache/taxonomic_similarities.pkl"},
}

print("Modality Feature Audit:")
for mod, paths in modalities.items():
    f_ok = os.path.exists(paths["feat"])
    s_ok = os.path.exists(paths["sim"])
    f_shape = np.load(paths["feat"]).shape if f_ok else "MISSING"
    print(f"  [{mod.upper():<10}] Features: {str(f_shape):<16} | Similarities Cached: {s_ok}")


--- 
## 4. Train and Evaluate All 4 Modalities
Runs `src.run_benchmark`: 4 modalities x 3 seeds, Strategy III plus all baselines, on CUDA if available.

In [ ]:
!python -m src.run_benchmark --seeds 42,123,456


--- 
## 5. Results Table (produced by the run above)

In [ ]:
from pathlib import Path
print(Path("results/RESULTS.md").read_text(encoding="utf-8"))


--- 
## 6. Download Results and Trained Models

In [ ]:
import os
from google.colab import files

out_zip = "savorlink_colab_results.zip"
!zip -r -q {out_zip} results/
print(f"Bundled results into {out_zip} ({os.path.getsize(out_zip) / 1e6:.2f} MB)")
files.download(out_zip)
